# `locityper_02_run_batch` — native Google Batch

Run **in a VWB JupyterLab app** after
[`locityper_00_prep_reference.ipynb`](locityper_00_prep_reference.ipynb).

This submits **minicram** then **genotype** with `gcloud batch`. No dsub
sidecars. Same Nearline `print_reads` + chr17 background interval as the
WDL, then `locityper preproc` / GNU-parallel-style per-locus genotype on
**one** VM (no WDL scatter).

This notebook is the **one-sample smoke** (toy CYP2D6 BED from prep). Cohort
cost + keep list + sharded dispatch live in
[`locityper_03_budget.ipynb`](locityper_03_budget.ipynb) and
[`locityper_04_dispatch.ipynb`](locityper_04_dispatch.ipynb). Status of every
Batch job: [`batch_monitor.ipynb`](batch_monitor.ipynb).

The WDL notebook ([`locityper_01_run_stream.ipynb`](locityper_01_run_stream.ipynb))
is unchanged. Docs: [`locityper/batch/README.md`](../../locityper/batch/README.md).

`SUBMIT_*` stay **off** until you turn one on. Submit genotype only after
minicram objects exist. VPC-SC cannot pull Docker Hub
(`eichlerlab/locityper:1.4.5.0`); set `LOCITYPER_DOCKER` to an Artifact
Registry mirror. The WGS CRAM stays a `gs://` env var.


In [10]:
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        script = p / "locityper" / "batch" / "submit_batch.py"
        if script.is_file():
            return p
        nested = p / "aou-lr-phase-2" / "locityper" / "batch" / "submit_batch.py"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError("Cannot find locityper/batch/submit_batch.py. Clone kvg/aou-lr-phase-2 and re-run.")


def sh(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check, text=True)


REPO_ROOT = find_repo_root()
BATCH_DIR = REPO_ROOT / "locityper" / "batch"
SCRATCH = Path.cwd() / "locityper_rw_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BATCH_DIR))
from submit_batch import build_genotype_job, build_minicram_job  # noqa: E402

OUTPUT_BUCKET_GS = os.environ.get("LT_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources/")
REGION = os.environ.get("DSUB_REGION", "us-central1")
SAMPLE_ID = os.environ.get("LOCITYPER_SAMPLE_ID", "1000000")
CRAM = os.environ.get(
    "LOCITYPER_CRAM",
    "gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram",
)
CRAI = os.environ.get(
    "LOCITYPER_CRAI",
    "gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram.crai",
)
REF_FA = os.environ.get(
    "LOCITYPER_REF_FA",
    "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta",
)
REF_FAI = os.environ.get(
    "LOCITYPER_REF_FAI",
    "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta.fai",
)
COUNTS_JF = os.environ.get(
    "LOCITYPER_COUNTS_JF",
    "gs://aou-lr-phase2-resources/locityper/refs/counts.Homo_sapiens_assembly38.k25.jf",
)
BED = os.environ.get("LOCITYPER_BED", "gs://aou-lr-phase2-resources/locityper/smoke.bed")
DB_TAR = os.environ.get("LOCITYPER_DB_TAR", "gs://aou-lr-phase2-resources/locityper/vcf_db.smoke.tar.gz")
SEX = os.environ.get("LOCITYPER_SEX", "female")
PRINT_READS_DOCKER = os.environ.get(
    "LOCITYPER_PRINT_READS_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-locityper-print-reads:0.1.2",
)
LOCITYPER_DOCKER = os.environ.get("LOCITYPER_DOCKER", "eichlerlab/locityper:1.4.5.0")
OUT_PREFIX = os.environ.get(
    "LT_BATCH_OUT_PREFIX",
    f"{OUTPUT_BUCKET_GS.rstrip('/')}/batchRuns/locityper",
)
MAX_RUN_DURATION = os.environ.get("LT_MAX_RUN_DURATION", "14400s")

SUBMIT_MINICRAM = False
SUBMIT_GENOTYPE = True
WATCH_JOB_ID = ""
JOB_ID_PATH = SCRATCH / "last_minicram.job_id"
GT_JOB_ID_PATH = SCRATCH / "last_genotype.job_id"

print("REPO_ROOT:", REPO_ROOT)
print("SAMPLE_ID:", SAMPLE_ID)
print("BED:", BED)
print("DB_TAR:", DB_TAR)
print("PRINT_READS_DOCKER:", PRINT_READS_DOCKER)
print("LOCITYPER_DOCKER:", LOCITYPER_DOCKER)
print("OUT_PREFIX:", OUT_PREFIX)
print("SUBMIT_MINICRAM:", SUBMIT_MINICRAM, "SUBMIT_GENOTYPE:", SUBMIT_GENOTYPE)


REPO_ROOT: /home/jupyter/repos/aou-lr-phase-2
SAMPLE_ID: 1000000
BED: gs://aou-lr-phase2-resources/locityper/smoke.bed
DB_TAR: gs://aou-lr-phase2-resources/locityper/vcf_db.smoke.tar.gz
PRINT_READS_DOCKER: us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-locityper-print-reads:0.1.2
LOCITYPER_DOCKER: eichlerlab/locityper:1.4.5.0
OUT_PREFIX: gs://aou-lr-phase2-resources/batchRuns/locityper
SUBMIT_MINICRAM: False SUBMIT_GENOTYPE: True


## PET SA / VPC

Fails here if this is not a VWB cloud app.


In [2]:
if shutil.which("gcloud") is None:
    raise SystemExit("gcloud is not on PATH. Open this notebook in a VWB Jupyter app.")
project = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()
sa = os.environ.get("PET_SA_EMAIL", "").strip()
if not project or not sa:
    raise SystemExit("GOOGLE_CLOUD_PROJECT / PET_SA_EMAIL empty. Use a VWB Jupyter app.")
print("project:", project)
print("PET_SA_EMAIL:", sa)
print("region:", REGION)


project: wb-steady-asparagus-6800
PET_SA_EMAIL: pet-2759429535691cc9a0dcb@wb-steady-asparagus-6800.iam.gserviceaccount.com
region: us-central1


## Per-sample CSV

Same columns as `locityper/configs/batch.header.csv`. Does not submit.


In [3]:
csv_path = SCRATCH / f"{SAMPLE_ID}.batch.csv"
with csv_path.open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(
        fh,
        fieldnames=[
            "sample_id", "cram", "crai", "ref_fa_uncompressed", "ref_fai_uncompressed",
            "counts_jf", "bed", "locityper_db_tar_gz",
        ],
    )
    w.writeheader()
    w.writerow({
        "sample_id": SAMPLE_ID,
        "cram": CRAM,
        "crai": CRAI,
        "ref_fa_uncompressed": REF_FA,
        "ref_fai_uncompressed": REF_FAI,
        "counts_jf": COUNTS_JF,
        "bed": BED,
        "locityper_db_tar_gz": DB_TAR,
    })
print("CSV:", csv_path)
print(csv_path.read_text())


CSV: /home/jupyter/repos/aou-lr-phase-2/notebooks/rw/locityper_rw_scratch/1000000.batch.csv
sample_id,cram,crai,ref_fa_uncompressed,ref_fai_uncompressed,counts_jf,bed,locityper_db_tar_gz
1000000,gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram,gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram.crai,gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta,gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta.fai,gs://aou-lr-phase2-resources/locityper/refs/counts.Homo_sapiens_assembly38.k25.jf,gs://aou-lr-phase2-resources/locityper/smoke.bed,gs://aou-lr-phase2-resources/locityper/vcf_db.smoke.tar.gz



## Dry-run Batch JSON

Always builds minicram + genotype job configs. No submit.


In [4]:
project = os.environ["GOOGLE_CLOUD_PROJECT"]
sa = os.environ["PET_SA_EMAIL"]
mini_job = build_minicram_job(
    csv_path=csv_path,
    worker_path=BATCH_DIR / "make_minicram.py",
    image=PRINT_READS_DOCKER,
    out_prefix=OUT_PREFIX,
    project=project,
    region=REGION,
    sa=sa,
    max_run_duration=MAX_RUN_DURATION,
)
gt_job = build_genotype_job(
    csv_path=csv_path,
    worker_path=BATCH_DIR / "genotype.py",
    image=LOCITYPER_DOCKER,
    out_prefix=OUT_PREFIX,
    project=project,
    region=REGION,
    sa=sa,
    max_run_duration=MAX_RUN_DURATION,
)
mini_cfg = SCRATCH / "minicram.job.json"
gt_cfg = SCRATCH / "genotype.job.json"
mini_cfg.write_text(json.dumps(mini_job, indent=2) + "\n")
gt_cfg.write_text(json.dumps(gt_job, indent=2) + "\n")
for label, job, cfg in (("minicram", mini_job, mini_cfg), ("genotype", gt_job, gt_cfg)):
    spec = job["taskGroups"][0]["taskSpec"]
    env = spec["environment"]["variables"]
    print(f"=== {label} ===", cfg, "bytes", cfg.stat().st_size)
    print("cpuMilli", spec["computeResource"], "runnables", len(spec["runnables"]))
    print("SAMPLE_ID", env.get("SAMPLE_ID"), "CRAM" if "CRAM" in env else "MINICRAM", env.get("CRAM") or env.get("MINICRAM"))
assert mini_job["allocationPolicy"]["network"]["networkInterfaces"][0]["noExternalIpAddress"] is True


=== minicram === /home/jupyter/repos/aou-lr-phase-2/notebooks/rw/locityper_rw_scratch/minicram.job.json bytes 19311
cpuMilli {'cpuMilli': 4000, 'memoryMib': 16384, 'bootDiskMib': 51200} runnables 1
SAMPLE_ID 1000000 CRAM gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram
=== genotype === /home/jupyter/repos/aou-lr-phase-2/notebooks/rw/locityper_rw_scratch/genotype.job.json bytes 20156
cpuMilli {'cpuMilli': 2000, 'memoryMib': 14336, 'bootDiskMib': 81920} runnables 3
SAMPLE_ID 1000000 MINICRAM gs://aou-lr-phase2-resources/batchRuns/locityper/1000000/1000000.minicram.cram


## Submit minicram

Set `SUBMIT_MINICRAM = True`. Native Batch, no dsub.


In [5]:
if not SUBMIT_MINICRAM:
    print("SUBMIT_MINICRAM is off")
else:
    job_id = f"lt-mini-{datetime.now(timezone.utc).strftime('%y%m%d-%H%M%S')}"
    sh([
        "gcloud", "batch", "jobs", "submit", job_id,
        f"--location={REGION}", f"--project={project}", f"--config={mini_cfg}",
    ])
    JOB_ID_PATH.write_text(job_id + "\n")
    print("submitted", job_id)


$ gcloud batch jobs submit lt-mini-260929-080945 --location=us-central1 --project=wb-steady-asparagus-6800 --config=/home/jupyter/repos/aou-lr-phase-2/notebooks/rw/locityper_rw_scratch/minicram.job.json


Job lt-mini-260929-080-3e0beeb8-995e-41ca0 was successfully submitted.


allocationPolicy:
  labels:
    batch-job-id: lt-mini-260929-080945
  location:
    allowedLocations:
    - regions/us-central1
    - zones/us-central1-a
    - zones/us-central1-b
    - zones/us-central1-c
    - zones/us-central1-f
  network:
    networkInterfaces:
    - network: projects/wb-steady-asparagus-6800/global/networks/network
      noExternalIpAddress: true
      subnetwork: projects/wb-steady-asparagus-6800/regions/us-central1/subnetworks/subnetwork
  serviceAccount:
    email: pet-2759429535691cc9a0dcb@wb-steady-asparagus-6800.iam.gserviceaccount.com
createTime: '2026-09-29T08:09:45.835044993Z'
logsPolicy:
  destination: CLOUD_LOGGING
name: projects/wb-steady-asparagus-6800/locations/us-central1/jobs/lt-mini-260929-080945
status:
  runDuration: 0s
  state: QUEUED
taskGroups:
- name: projects/870566182483/locations/us-central1/jobs/lt-mini-260929-080945/taskGroups/group0
  parallelism: '1'
  taskCount: '1'
  taskSpec:
    computeResource:
      bootDiskMib: '51200'
      cp

## Watch / list outputs


In [ ]:
job_id = WATCH_JOB_ID.strip() or (JOB_ID_PATH.read_text().strip() if JOB_ID_PATH.is_file() else "")
if not job_id:
    print("no job id yet")
else:
    sh(["gcloud", "batch", "jobs", "describe", job_id, f"--location={REGION}", f"--project={project}", "--format=yaml(status.state,status.statusEvents)"])
print("outputs under", f"{OUT_PREFIX.rstrip('/')}/{SAMPLE_ID}/")
sh(["gcloud", "storage", "ls", f"{OUT_PREFIX.rstrip('/')}/{SAMPLE_ID}/"], check=False)


$ gcloud batch jobs describe lt-mini-260929-080945 --location=us-central1 --project=wb-steady-asparagus-6800 --format=yaml(status.state,status.statusEvents)
status:
  state: SUCCEEDED
  statusEvents:
  - description: Job state is set from QUEUED to SCHEDULED for job projects/870566182483/locations/us-central1/jobs/lt-mini-260929-080945.
    eventTime: '2026-09-29T08:09:52.119822834Z'
    type: STATUS_CHANGED
  - description: Job state is set from SCHEDULED to RUNNING for job projects/870566182483/locations/us-central1/jobs/lt-mini-260929-080945.
    eventTime: '2026-09-29T08:11:23.020359228Z'
    type: STATUS_CHANGED
  - description: Job state is set from RUNNING to SUCCEEDED for job projects/870566182483/locations/us-central1/jobs/lt-mini-260929-080945.
    eventTime: '2026-09-29T08:12:03.431478231Z'
    type: STATUS_CHANGED
outputs under gs://aou-lr-phase2-resources/batchRuns/locityper/1000000/
$ gcloud storage ls gs://aou-lr-phase2-resources/batchRuns/locityper/1000000/


## Submit genotype

Set `SUBMIT_GENOTYPE = True` after minicram objects exist.


In [11]:
if not SUBMIT_GENOTYPE:
    print("SUBMIT_GENOTYPE is off")
else:
    job_id = f"lt-gt-{datetime.now(timezone.utc).strftime('%y%m%d-%H%M%S')}"
    sh([
        "gcloud", "batch", "jobs", "submit", job_id,
        f"--location={REGION}", f"--project={project}", f"--config={gt_cfg}",
    ])
    GT_JOB_ID_PATH.write_text(job_id + "\n")
    print("submitted", job_id)


$ gcloud batch jobs submit lt-gt-260929-081453 --location=us-central1 --project=wb-steady-asparagus-6800 --config=/home/jupyter/repos/aou-lr-phase-2/notebooks/rw/locityper_rw_scratch/genotype.job.json


Job lt-gt-260929-08145-8a13f4a7-5044-4c050 was successfully submitted.


allocationPolicy:
  labels:
    batch-job-id: lt-gt-260929-081453
  location:
    allowedLocations:
    - regions/us-central1
    - zones/us-central1-a
    - zones/us-central1-b
    - zones/us-central1-c
    - zones/us-central1-f
  network:
    networkInterfaces:
    - network: projects/wb-steady-asparagus-6800/global/networks/network
      noExternalIpAddress: true
      subnetwork: projects/wb-steady-asparagus-6800/regions/us-central1/subnetworks/subnetwork
  serviceAccount:
    email: pet-2759429535691cc9a0dcb@wb-steady-asparagus-6800.iam.gserviceaccount.com
createTime: '2026-09-29T08:14:54.614228898Z'
logsPolicy:
  destination: CLOUD_LOGGING
name: projects/wb-steady-asparagus-6800/locations/us-central1/jobs/lt-gt-260929-081453
status:
  runDuration: 0s
  state: QUEUED
taskGroups:
- name: projects/870566182483/locations/us-central1/jobs/lt-gt-260929-081453/taskGroups/group0
  parallelism: '1'
  taskCount: '1'
  taskSpec:
    computeResource:
      bootDiskMib: '81920'
      cpuMilli

In [13]:
job_id

'lt-mini-260929-080945'